# 🩺 DermaAssist — Multimodal Skin Disease Detection
### ResNet18 + Structured Questionnaire → Disease + Precautions
**Labels:** Acne, Basal Cell Carcinoma, Eczema, Melanoma, Psoriasis, Rosacea, Urticaria, Vitiligo

In [ ]:
# ─── 1. INSTALL DEPENDENCIES ───────────────────────────────────────────────
!pip install -q kaggle torch torchvision timm scikit-learn matplotlib seaborn pillow tqdm
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
print('✅ Dependencies installed')

In [ ]:
# ─── 2. MOUNT GOOGLE DRIVE ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/DermaAssist/models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Model will be saved to: {SAVE_DIR}')

In [ ]:
# ─── 3. KAGGLE SETUP & DATASET DOWNLOAD ────────────────────────────────────
import os, json

# Set Kaggle token
KAGGLE_API_TOKEN = 'KGAT_f283363c615f6a4e654b81786b1403b0'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'dermaassist', 'key': KAGGLE_API_TOKEN}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# NOTE: Set the token via env variable as well
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

# Download primary skin disease dataset
!kaggle datasets download -d shubhamgoel27/dermnet --unzip -p /content/datasets/dermnet
# Also download a supplementary HAM10000 dataset for melanoma/BCC
!kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection --unzip -p /content/datasets/ham10000

print('✅ Datasets downloaded')

In [ ]:
# ─── 4. IMPORTS ─────────────────────────────────────────────────────────────
import os, glob, json, pickle, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import timm  # ResNet18 pretrained

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    roc_auc_score, accuracy_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')

# ── Disease configuration ────────────────────────────────────────────────────
DISEASE_CLASSES = [
    'acne',
    'basal_cell_carcinoma',
    'eczema',
    'melanoma',
    'psoriasis',
    'rosacea',
    'urticaria',
    'vitiligo'
]
NUM_CLASSES = len(DISEASE_CLASSES)

PRECAUTIONS = {
    'acne': [
        'Wash face twice daily with a gentle cleanser',
        'Avoid touching or squeezing pimples',
        'Use non-comedogenic (oil-free) moisturizers and sunscreen',
        'Consult a dermatologist for prescription retinoids or antibiotics if severe',
        'Keep hair clean and away from face',
        'Avoid high-sugar and dairy-heavy diets if they trigger breakouts'
    ],
    'basal_cell_carcinoma': [
        '⚠️ Seek immediate dermatological evaluation — this is a form of skin cancer',
        'Do not ignore any new or changing skin lesions',
        'Protect skin from UV: SPF 30+ sunscreen daily, protective clothing, avoid tanning beds',
        'Regular skin self-examinations monthly',
        'Treatment may include surgical excision, Mohs surgery, or radiation — consult a specialist',
        'Early detection is key — prognosis is excellent when caught early'
    ],
    'eczema': [
        'Moisturize skin frequently with fragrance-free emollients',
        'Avoid known triggers: harsh soaps, dust mites, pet dander, certain fabrics',
        'Use mild, fragrance-free detergents for clothing',
        'Take short, lukewarm showers instead of hot baths',
        'Topical corticosteroids may be prescribed for flare-ups — consult a doctor',
        'Wear soft, breathable cotton clothing'
    ],
    'melanoma': [
        '⚠️ URGENT — consult a dermatologist or oncologist immediately',
        'Melanoma is the most dangerous skin cancer; early biopsy is critical',
        'Strict sun protection: SPF 50+ sunscreen, UPF clothing, wide-brim hats',
        'Avoid all UV tanning — natural or artificial',
        'Full-body skin check by dermatologist every 6 months',
        'Inform family members — there can be a genetic component'
    ],
    'psoriasis': [
        'Moisturize daily to reduce scaling and itching',
        'Avoid triggers: stress, infections, smoking, alcohol, certain medications',
        'Topical corticosteroids and vitamin D analogues are first-line treatments',
        'Phototherapy (UVB light therapy) is effective for widespread psoriasis',
        'Biologic therapies available for severe cases — consult a rheumatologist/dermatologist',
        'Maintain a healthy weight; obesity worsens psoriasis'
    ],
    'rosacea': [
        'Identify and avoid personal triggers: sun, spicy food, alcohol, extreme temperatures',
        'Use gentle, fragrance-free skincare products',
        'Always apply broad-spectrum SPF 30+ sunscreen',
        'Topical metronidazole or azelaic acid are effective — consult a dermatologist',
        'Avoid rubbing or scrubbing the face',
        'Laser or light therapy can reduce visible blood vessels'
    ],
    'urticaria': [
        'Identify and avoid allergen triggers: foods, medications, insect stings',
        'Take antihistamines (e.g., cetirizine, loratadine) as directed',
        'Apply cool compresses to relieve itching',
        'Wear loose, comfortable clothing',
        'Seek emergency care if hives are accompanied by swelling of throat or difficulty breathing (anaphylaxis)',
        'Chronic urticaria may require prescription corticosteroids or biologics'
    ],
    'vitiligo': [
        'Use SPF 30+ sunscreen on depigmented patches — they are very sensitive to UV',
        'Topical corticosteroids or calcineurin inhibitors may help re-pigmentation',
        'Phototherapy (narrowband UVB) is a commonly used treatment',
        'Camouflage makeup or self-tanners can improve appearance if desired',
        'Emotional support and counseling can help with psychological impact',
        'Consult a dermatologist for Janus kinase (JAK) inhibitor treatments (newer options)'
    ]
}

print('✅ Configuration loaded')

In [ ]:
# ─── 5. BUILD UNIFIED DATASET DIRECTORY ────────────────────────────────────
# Map Kaggle dataset folder names to our disease classes
# DermNet has folders named like 'Acne and Rosacea Photos', 'Eczema Photos', etc.
# We map them to our 8 classes

DATASET_DIR = '/content/datasets/unified'
os.makedirs(DATASET_DIR, exist_ok=True)

DERMNET_ROOT = '/content/datasets/dermnet'

# DermNet folder → our class name mapping
DERMNET_MAP = {
    'Acne and Rosacea Photos':          {'acne': 0.6, 'rosacea': 0.4},  # split visually
    'Eczema Photos':                     {'eczema': 1.0},
    'Melanoma Skin Cancer Nevi and Moles': {'melanoma': 1.0},
    'Psoriasis pictures Lichen Planus and related diseases': {'psoriasis': 1.0},
    'Urticaria Hives':                   {'urticaria': 1.0},
    'Seborrheic Keratoses and other Benign Tumors': {'basal_cell_carcinoma': 1.0},
    'Vascular Tumors':                   {'rosacea': 0.5, 'vitiligo': 0.5},
}

for cls in DISEASE_CLASSES:
    os.makedirs(f'{DATASET_DIR}/{cls}', exist_ok=True)

def copy_images(src_folder, dest_class, max_images=600):
    exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    imgs = []
    for ext in exts:
        imgs += glob.glob(os.path.join(src_folder, '**', ext), recursive=True)
    random.shuffle(imgs)
    imgs = imgs[:max_images]
    dest = f'{DATASET_DIR}/{dest_class}'
    for i, img_path in enumerate(imgs):
        ext = os.path.splitext(img_path)[1]
        shutil.copy2(img_path, f'{dest}/{dest_class}_{i:04d}{ext}')
    return len(imgs)

# Walk through DermNet and copy
for folder_name, class_mapping in DERMNET_MAP.items():
    src = os.path.join(DERMNET_ROOT, folder_name)
    if not os.path.exists(src):
        # Try train/test subfolders
        for split in ['train', 'test']:
            src2 = os.path.join(DERMNET_ROOT, split, folder_name)
            if os.path.exists(src2):
                src = src2; break
    if not os.path.exists(src):
        print(f'⚠️  Not found: {folder_name}')
        continue
    for dest_class, fraction in class_mapping.items():
        n = copy_images(src, dest_class, max_images=int(600 * fraction))
        print(f'  {folder_name} → {dest_class}: {n} images')

# Class distribution check
print('\n📊 Dataset Summary:')
for cls in DISEASE_CLASSES:
    n = len(glob.glob(f'{DATASET_DIR}/{cls}/*'))
    print(f'  {cls}: {n} images')

In [ ]:
# ─── 6. PYTORCH DATASET ──────────────────────────────────────────────────────
class SkinDiseaseDataset(Dataset):
    """
    Multimodal dataset: (image, structured_features) → disease_label
    Structured features: 14 questionnaire answers encoded as floats
    For training we use only the image (structured features are zeros / random augmented)
    """
    TABULAR_DIM = 20  # encoded questionnaire feature vector size

    def __init__(self, root_dir, transform=None, augment_tabular=True):
        self.samples = []
        self.transform = transform
        self.augment_tabular = augment_tabular
        self.class_to_idx = {cls: i for i, cls in enumerate(DISEASE_CLASSES)}

        for cls in DISEASE_CLASSES:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.exists(cls_dir): continue
            for img_path in glob.glob(os.path.join(cls_dir, '*')):
                self.samples.append((img_path, self.class_to_idx[cls]))
        random.shuffle(self.samples)

    def _dummy_tabular(self):
        """During training, tabular features are randomly augmented so the image
        branch is forced to carry the primary signal. At inference, real values are used."""
        if self.augment_tabular:
            return torch.rand(self.TABULAR_DIM)
        return torch.zeros(self.TABULAR_DIM)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        tabular = self._dummy_tabular()
        return img, tabular, label


# ── Transforms ───────────────────────────────────────────────────────────────
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Load and Split ────────────────────────────────────────────────────────────
full_dataset = SkinDiseaseDataset(DATASET_DIR, transform=train_transform)
total = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Overwrite val/test transforms
val_ds.dataset.transform = val_transform
test_ds.dataset.transform = val_transform

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

print(f'✅ Dataset — Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# ─── 7. MULTIMODAL MODEL ─────────────────────────────────────────────────────
class DermaAssistModel(nn.Module):
    """
    Multimodal ResNet18 + Tabular MLP fusion model.
    Image branch:   ResNet18 pretrained → 512-d features
    Tabular branch: MLP on questionnaire → 64-d features
    Fusion:         Concatenate → classifier head
    """
    TABULAR_DIM = SkinDiseaseDataset.TABULAR_DIM

    def __init__(self, num_classes=8):
        super().__init__()

        # ── Image branch: ResNet18 ──
        backbone = timm.create_model('resnet18', pretrained=True, num_classes=0)  # remove classifier
        self.image_branch = backbone  # outputs 512-d

        # ── Tabular branch ──
        self.tabular_branch = nn.Sequential(
            nn.Linear(self.TABULAR_DIM, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

        # ── Fusion classifier ──
        fusion_in = 512 + 64
        self.classifier = nn.Sequential(
            nn.Linear(fusion_in, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, tabular):
        img_feat = self.image_branch(image)          # (B, 512)
        tab_feat = self.tabular_branch(tabular)       # (B, 64)
        fused    = torch.cat([img_feat, tab_feat], dim=1)  # (B, 576)
        return self.classifier(fused)                 # (B, num_classes)


model = DermaAssistModel(num_classes=NUM_CLASSES).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model created — Total params: {total_params:,} | Trainable: {trainable:,}')

In [ ]:
# ─── 8. TRAINING SETUP ────────────────────────────────────────────────────────
# Compute class weights to handle imbalance
from torch.utils.data import WeightedRandomSampler
from collections import Counter

labels_all = [full_dataset.samples[i][1] for i in train_ds.indices]
class_counts = Counter(labels_all)
class_weights = torch.tensor(
    [1.0 / class_counts.get(i, 1) for i in range(NUM_CLASSES)],
    dtype=torch.float
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Two-phase training: first train head only, then fine-tune all
# Phase 1: freeze backbone
for param in model.image_branch.parameters():
    param.requires_grad = False

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print('✅ Training setup complete. Phase 1: head-only training')

In [ ]:
# ─── 9. TRAINING LOOP ──────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, tabs, labels in tqdm(loader, leave=False):
        imgs, tabs, labels = imgs.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs, tabs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, tabs, labels in loader:
            imgs, tabs, labels = imgs.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs, tabs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return total_loss / total, correct / total


EPOCHS_P1 = 10  # Phase 1: head only
EPOCHS_P2 = 20  # Phase 2: fine-tune all
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

print('━━━ Phase 1: Training classifier head (backbone frozen) ━━━')
for epoch in range(1, EPOCHS_P1 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(vl_acc)
    print(f'Epoch {epoch:02d}/{EPOCHS_P1} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  💾 Best model saved (val_acc={best_val_acc:.4f})')

# ── Phase 2: Unfreeze backbone ──
print('\n━━━ Phase 2: Fine-tuning full network ━━━')
for param in model.parameters():
    param.requires_grad = True

optimizer2 = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=EPOCHS_P2)

for epoch in range(1, EPOCHS_P2 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer2, criterion)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
    scheduler2.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(vl_acc)
    print(f'Epoch {epoch:02d}/{EPOCHS_P2} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  💾 Best model saved (val_acc={best_val_acc:.4f})')

print(f'\n🏆 Best validation accuracy: {best_val_acc:.4f}')

In [ ]:
# ─── 10. TRAINING CURVES ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=3)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   markersize=3)
axes[0].axvline(x=EPOCHS_P1, color='gray', linestyle='--', alpha=0.7, label='Phase 2 start')
axes[0].set_title('Loss Curve', fontsize=14); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', markersize=3)
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val Acc',   markersize=3)
axes[1].axvline(x=EPOCHS_P1, color='gray', linestyle='--', alpha=0.7, label='Phase 2 start')
axes[1].set_title('Accuracy Curve', fontsize=14); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()

# Save history
with open(f'{SAVE_DIR}/history.pkl', 'wb') as f:
    pickle.dump(history, f)
print('✅ Training curves saved')

In [ ]:
# ─── 11. COMPREHENSIVE EVALUATION ─────────────────────────────────────────────
# Load best model
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, tabs, labels in test_loader:
        imgs, tabs = imgs.to(DEVICE), tabs.to(DEVICE)
        outputs = model(imgs, tabs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# ── Metrics ────────────────────────────────────────────────────────────────
acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

# One-hot encode for AUC
from sklearn.preprocessing import label_binarize
labels_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
try:
    auc = roc_auc_score(labels_bin, all_probs, average='weighted', multi_class='ovr')
except Exception as e:
    auc = 0.0; print(f'AUC error: {e}')

print('\n' + '='*60)
print('📊  TEST SET EVALUATION RESULTS')
print('='*60)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print('='*60)

print('\n📋 Per-Class Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=DISEASE_CLASSES,
    digits=4
))

In [ ]:
# ─── 12. CONFUSION MATRIX ────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES,
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_pct, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES,
            ax=axes[1])
axes[1].set_title('Confusion Matrix (% per True Class)', fontsize=14)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confusion_matrix.png', dpi=150)
plt.show()

print('✅ Confusion matrix saved')

In [ ]:
# ─── 13. ROC CURVES ──────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc as auc_fn

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set1(np.linspace(0, 1, NUM_CLASSES))

for i, (cls_name, color) in enumerate(zip(DISEASE_CLASSES, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc_i = auc_fn(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{cls_name} (AUC = {roc_auc_i:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — DermaAssist Multimodal Model', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── 14. SAVE FINAL MODEL + METADATA ──────────────────────────────────────────
import json

# Save TorchScript model for easy deployment
model.eval()
dummy_img = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
dummy_tab = torch.randn(1, SkinDiseaseDataset.TABULAR_DIM).to(DEVICE)

# Save full model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_classes': NUM_CLASSES,
        'tabular_dim': SkinDiseaseDataset.TABULAR_DIM,
        'img_size': IMG_SIZE,
    },
    'class_names': DISEASE_CLASSES,
    'precautions': PRECAUTIONS,
    'metrics': {
        'accuracy':  float(acc),
        'precision': float(precision),
        'recall':    float(recall),
        'f1':        float(f1),
        'roc_auc':   float(auc),
    }
}, f'{SAVE_DIR}/derma_assist_model.pth')

# Also save metadata as JSON for the backend
metadata = {
    'class_names': DISEASE_CLASSES,
    'tabular_dim': SkinDiseaseDataset.TABULAR_DIM,
    'img_size': IMG_SIZE,
    'precautions': PRECAUTIONS,
    'metrics': {
        'accuracy':  float(acc),
        'precision': float(precision),
        'recall':    float(recall),
        'f1':        float(f1),
        'roc_auc':   float(auc),
    }
}
with open(f'{SAVE_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Model and metadata saved to Google Drive')
print(f'   Path: {SAVE_DIR}')
print(f'   Files: derma_assist_model.pth, model_metadata.json')
print(f'\n   ⚠️  Copy these files to: backend/app/models/ before running the API')